# Phase 6.2: Task-Aware Dual-Matching Diffusion

This notebook trains your Conditional 1D DDPM but adds a **Task-Aware Clinical Loss** term based on the TDD (2025) paper. 
It is optimized to use **Multiple GPUs via DataParallel** and includes the fixed evaluation code.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import ast
import requests
import io
import math
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

In [ ]:
# --- 1. Load Data and Labels ---
ptbxl_url = "https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv"
scp_url = "https://physionet.org/files/ptb-xl/1.0.3/scp_statements.csv"
ptbxl_df = pd.read_csv(io.StringIO(requests.get(ptbxl_url).text))
scp_df = pd.read_csv(io.StringIO(requests.get(scp_url).text), index_col=0)

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in scp_df.index:
            cls = scp_df.loc[key].diagnostic_class
            if str(cls) != 'nan':
                tmp.append(cls)
    return list(set(tmp))

ptbxl_df['scp_codes'] = ptbxl_df['scp_codes'].apply(lambda x: ast.literal_eval(x))
ptbxl_df['diagnostic_superclass'] = ptbxl_df['scp_codes'].apply(aggregate_diagnostic)

# Defaulting paths to current directory (Upload to Kaggle working dir or Kaggle dataset)
metadata_path = 'metadata.csv' 
clean_path = 'clean_samples.npy'
noisy_path = 'noisy_samples.npy'

metadata = pd.read_csv(metadata_path)
clean_all = np.load(clean_path)
noisy_all = np.load(noisy_path)

metadata['ecg_id_int'] = metadata['ecg_id'].astype(int)
merged = metadata.merge(ptbxl_df[['ecg_id', 'diagnostic_superclass']], left_on='ecg_id_int', right_on='ecg_id', how='left')
valid_indices = merged['diagnostic_superclass'].apply(lambda x: isinstance(x, list) and len(x) > 0)

clean_filtered = clean_all[valid_indices]
noisy_filtered = noisy_all[valid_indices]
labels_raw = merged['diagnostic_superclass'][valid_indices].tolist()
ecg_ids_filtered = merged['ecg_id_int'][valid_indices].values

superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
y = np.zeros((len(labels_raw), 5))
for i, labels in enumerate(labels_raw):
    for j, sc in enumerate(superclasses):
        if sc in labels:
            y[i, j] = 1

if len(clean_filtered.shape) == 2:
    clean_filtered = np.expand_dims(clean_filtered, axis=1)
    noisy_filtered = np.expand_dims(noisy_filtered, axis=1)

# Use GroupShuffleSplit to prevent data leakage across leads of the same patient
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(clean_filtered, y, groups=ecg_ids_filtered))

X_train, X_val = clean_filtered[train_idx], clean_filtered[val_idx]
cond_train, cond_val = noisy_filtered[train_idx], noisy_filtered[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

In [ ]:
# --- 2. Load the Frozen Oracle with DataParallel ---
class ResNet1DBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.downsample = nn.Sequential(nn.Conv1d(in_channels, out_channels, 1, stride), nn.BatchNorm1d(out_channels)) if stride != 1 or in_channels != out_channels else nn.Identity()
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.downsample(x)
        return self.relu(out)

class ResNet1D(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Conv1d(1, 32, 15, stride=2, padding=7), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(3, 2, 1))
        self.layer2 = ResNet1DBlock(32, 64, stride=2)
        self.layer3 = ResNet1DBlock(64, 128, stride=2)
        self.layer4 = ResNet1DBlock(128, 256, stride=2)
        self.layer5 = ResNet1DBlock(256, 512, stride=2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.pool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x)

oracle = ResNet1D(num_classes=5)

# Handle loading whether saved with DataParallel or not
state_dict = torch.load('best_clinical_oracle.pth', map_location='cpu')
if list(state_dict.keys())[0].startswith('module.'):
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
oracle.load_state_dict(state_dict)

if torch.cuda.device_count() > 1:
    oracle = nn.DataParallel(oracle)

oracle = oracle.to(device)
oracle.eval()
for param in oracle.parameters():
    param.requires_grad = False  # Freeze the oracle
print("Frozen Clinical Oracle Loaded & Moved to Device(s).")

In [ ]:
# --- 3. Diffusion Definitions ---
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=time.device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

class Block1D(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim):
        super().__init__()
        self.mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_channels))
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=1), nn.GroupNorm(8, out_channels), nn.SiLU(),
            nn.Conv1d(out_channels, out_channels, 3, padding=1), nn.GroupNorm(8, out_channels), nn.SiLU()
        )
    def forward(self, x, t):
        return self.conv(x) + self.mlp(t).unsqueeze(-1)

class ConditionalUNet1D(nn.Module):
    def __init__(self, channels=1, time_emb_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(SinusoidalPositionEmbeddings(time_emb_dim), nn.Linear(time_emb_dim, time_emb_dim), nn.SiLU())
        self.init_conv = nn.Conv1d(channels * 2, 64, 7, padding=3)
        self.downs = nn.ModuleList([Block1D(64, 128, time_emb_dim), Block1D(128, 256, time_emb_dim)])
        self.pool = nn.MaxPool1d(2)
        self.mid = Block1D(256, 256, time_emb_dim)
        self.ups = nn.ModuleList([Block1D(256 + 256, 128, time_emb_dim), Block1D(128 + 128, 64, time_emb_dim)])
        self.up_sample = nn.Upsample(scale_factor=2, mode='linear', align_corners=False)
        self.final_conv = nn.Conv1d(64, channels, 3, padding=1)
    def forward(self, x, t, cond):
        t = self.time_mlp(t)
        x = torch.cat([x, cond], dim=1)
        x = self.init_conv(x)
        skip_connections = []
        for down in self.downs:
            x = down(x, t)
            skip_connections.append(x)
            x = self.pool(x)
        x = self.mid(x, t)
        for up in self.ups:
            x = self.up_sample(x)
            skip_x = skip_connections.pop()
            if x.shape[-1] != skip_x.shape[-1]:
                x = F.pad(x, (0, skip_x.shape[-1] - x.shape[-1]))
            x = torch.cat((x, skip_x), dim=1)
            x = up(x, t)
        return self.final_conv(x)

T = 500
betas = torch.linspace(0.0001, 0.02, T).to(device)
alphas = 1. - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - alphas_cumprod)

def gather(consts, t, x_shape):
    out = consts.gather(-1, t)
    return out.reshape(t.shape[0], *((1,) * (len(x_shape) - 1)))

In [ ]:
# --- 4. Task-Aware Training Loop --- 
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), 
                              torch.tensor(cond_train, dtype=torch.float32), 
                              torch.tensor(y_train, dtype=torch.float32))
# Increased batch size to saturate 2 GPUs
batch_size = 128 if torch.cuda.device_count() > 1 else 32 
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model = ConditionalUNet1D(channels=1)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
mse_criterion = nn.MSELoss()
bce_criterion = nn.BCEWithLogitsLoss() 
lambda_clinical = 0.5  # Weight of the clinical loss

epochs = 200
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

print(f"Starting Task-Aware Diffusion Training with Batch Size {batch_size}...")
for epoch in range(1, epochs + 1):
    model.train()
    total_mse = 0
    total_bce = 0
    
    for clean_b, cond_b, y_b in train_loader:
        clean_b, cond_b, y_b = clean_b.to(device), cond_b.to(device), y_b.to(device)
        
        # 1. Forward Diffusion
        t = torch.randint(0, T, (clean_b.shape[0],), device=device).long()
        eps = torch.randn_like(clean_b)
        x_t = gather(sqrt_alphas_cumprod, t, clean_b.shape) * clean_b + gather(sqrt_one_minus_alphas_cumprod, t, clean_b.shape) * eps
        
        # 2. Predict Noise
        eps_pred = model(x_t, t, cond_b)
        loss_mse = mse_criterion(eps_pred, eps)
        
        # 3. TASK-AWARE LOSS (Predict x0 mathematically and classify it)
        sqrt_acp_t = gather(sqrt_alphas_cumprod, t, clean_b.shape)
        sqrt_1m_acp_t = gather(sqrt_one_minus_alphas_cumprod, t, clean_b.shape)
        x0_pred = (x_t - sqrt_1m_acp_t * eps_pred) / sqrt_acp_t
        
        # Pass the predicted clean signal through the Oracle
        logits_pred = oracle(x0_pred)
        loss_bce = bce_criterion(logits_pred, y_b)
        
        # Total Loss
        loss = loss_mse + lambda_clinical * loss_bce
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_mse += loss_mse.item()
        total_bce += loss_bce.item()
        
    scheduler.step()
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch}/{epochs} | MSE: {total_mse/len(train_loader):.4f} | Clinical BCE: {total_bce/len(train_loader):.4f}")

torch.save(model.module.state_dict() if hasattr(model, 'module') else model.state_dict(), 'best_task_aware_diffusion.pth')
print("Training Complete! Saved best_task_aware_diffusion.pth")

In [ ]:
# --- 5. Partial Reverse Diffusion Inference ---
def denoise_sample_partial(model, cond, t_start=30):
    """
    Performs partial reverse diffusion to act as a refinement layer.
    We add t_start noise to the condition, then reverse diffuse.
    """
    model.eval()
    with torch.no_grad():
        t_start_tensor = torch.full((cond.shape[0],), t_start, device=cond.device, dtype=torch.long)
        noise = torch.randn_like(cond)
        
        sqrt_acp_t = gather(sqrt_alphas_cumprod, t_start_tensor, cond.shape)
        sqrt_1m_acp_t = gather(sqrt_one_minus_alphas_cumprod, t_start_tensor, cond.shape)
        
        # Start with noisy observation + scaled noise
        x = sqrt_acp_t * cond + sqrt_1m_acp_t * noise
        
        # Reverse diffusion from t_start down to 0
        for i in reversed(range(t_start)):
            t = torch.full((cond.shape[0],), i, device=cond.device, dtype=torch.long)
            eps_pred = model(x, t, cond)
            
            alpha_t = alphas[i]
            alpha_bar_t = alphas_cumprod[i]
            beta_t = betas[i]
            
            if i > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)
                
            x = (1 / torch.sqrt(alpha_t)) * (x - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * eps_pred) + torch.sqrt(beta_t) * noise
            
    return x

In [ ]:
# --- 6. Evaluation on Validation/Test Set ---
print("Evaluating the Task-Aware Diffusion Model on the Test Set...")

# 1. Generate Denoised test set in batches to save memory
denoised_val = []
eval_batch_size = 128 if torch.cuda.device_count() > 1 else 32
eval_loader = DataLoader(torch.tensor(cond_val, dtype=torch.float32), batch_size=eval_batch_size, shuffle=False)

for cond_b in eval_loader:
    cond_b = cond_b.to(device)
    denoised_b = denoise_sample_partial(model, cond_b, t_start=30)
    denoised_val.append(denoised_b.cpu().numpy())
denoised_val = np.concatenate(denoised_val, axis=0)

# 2. Evaluation function using the Oracle
def evaluate_with_oracle(data_arrays, labels_arrays):
    oracle.eval()
    data_tensor = torch.tensor(data_arrays, dtype=torch.float32)
    labels_tensor = torch.tensor(labels_arrays, dtype=torch.float32)
    dataset = TensorDataset(data_tensor, labels_tensor)
    loader = DataLoader(dataset, batch_size=eval_batch_size, shuffle=False)
    
    all_probs = []
    with torch.no_grad():
        for x_b, _ in loader:
            x_b = x_b.to(device)
            logits = oracle(x_b)
            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
            
    all_probs = np.concatenate(all_probs, axis=0)
    preds = (all_probs > 0.5).astype(int)
    
    # Now using the correctly imported functions
    macro_f1 = f1_score(labels_arrays, preds, average='macro', zero_division=0)
    auroc = roc_auc_score(labels_arrays, all_probs, average='macro')
    return macro_f1, auroc

# 3. Run metrics
clean_f1, clean_auroc = evaluate_with_oracle(X_val, y_val)
noisy_f1, noisy_auroc = evaluate_with_oracle(cond_val, y_val)
denoised_f1, denoised_auroc = evaluate_with_oracle(denoised_val, y_val)

print(f"\n--- Clinical Diagnostic Retention (Oracle Evaluation) ---")
print(f"Clean Ground Truth -> Macro F1: {clean_f1:.4f}, AUROC: {clean_auroc:.4f}")
print(f"Noisy Digitization -> Macro F1: {noisy_f1:.4f}, AUROC: {noisy_auroc:.4f}")
print(f"Task-Aware Denoised-> Macro F1: {denoised_f1:.4f}, AUROC: {denoised_auroc:.4f}")

diff_f1 = denoised_f1 - noisy_f1
print(f"\nImprovement over Noisy: {diff_f1:+.4f} Macro F1")